<p style="text-align:center;">
<img src="https://github.com/digital-futures-academy/DataScienceMasterResources/blob/main/Resources/datascience-notebook-header.png?raw=true"
     alt="DigitalFuturesLogo"
     style="float: center; margin-right: 10px;" />
</p>

# Digital Futures Frontier AI: API Fundamentals (Part 1)

**APIs (Application Programming Interfaces)** are how we can connect our applications to external data sources and programmatically connect to services. 

**In this workshop, you will:**
1.  Understand the Request/Response cycle.
2.  Master `JSON` (the language of AI agents).
3.  Perform **Chained Retrieval** (fetching data that links to other data).
4.  Format raw data into **Context** for an AI.

In [2]:
import requests
from pprint import pprint
import time

## 1. The Basic GET Request
We are using the **PokeAPI**. It's stable, free, and a well structured API..

In the code below, we are using the requests library to make a GET request to the PokeAPI.

In [3]:
# The Base URL is the "Root" of the API
base_url = "https://pokeapi.co/api/v2"
endpoint = "/pokemon/pikachu"

# Construct the full URL
full_url = f"{base_url}{endpoint}"

# Send the GET request
response = requests.get(full_url)

print(f"Status Code: {response.status_code}") # 200 means OK, 404 means Not Found
print(f"URL Used: {response.url}")

data = response.json()

Status Code: 200
URL Used: https://pokeapi.co/api/v2/pokemon/pikachu


We can "pretty print" the JSON to view the layout of the data.

In [3]:
# EX

pprint(data)

{'abilities': [{'ability': {'name': 'static',
                            'url': 'https://pokeapi.co/api/v2/ability/9/'},
                'is_hidden': False,
                'slot': 1},
               {'ability': {'name': 'lightning-rod',
                            'url': 'https://pokeapi.co/api/v2/ability/31/'},
                'is_hidden': True,
                'slot': 3}],
 'base_experience': 112,
 'cries': {'latest': 'https://raw.githubusercontent.com/PokeAPI/cries/main/cries/pokemon/latest/25.ogg',
           'legacy': 'https://raw.githubusercontent.com/PokeAPI/cries/main/cries/pokemon/legacy/25.ogg'},
 'forms': [{'name': 'pikachu',
            'url': 'https://pokeapi.co/api/v2/pokemon-form/25/'}],
 'game_indices': [{'game_index': 84,
                   'version': {'name': 'red',
                               'url': 'https://pokeapi.co/api/v2/version/1/'}},
                  {'game_index': 84,
                   'version': {'name': 'blue',
                               'url': '

## 2. Handling the JSON Body
The response contains a massive amount of data. Dumping this entire JSON into an LLM prompt is expensive and wasteful. You need to learn to **parse** and **extract** only the relevant data.

Task: Print out the available keys from the JSON body.

In [4]:
# EX
    
print(f"Available Keys: {list(data.keys())}")

Available Keys: ['id', 'name', 'base_experience', 'height', 'is_default', 'order', 'weight', 'abilities', 'past_abilities', 'forms', 'game_indices', 'held_items', 'location_area_encounters', 'moves', 'species', 'sprites', 'cries', 'stats', 'past_stats', 'types', 'past_types']


Task: Print the name, weight and types of Pokemon from the Pokemon API. 

There may be more than one type so we need to account for that.

In [5]:
# EX

name = data['name']
types = [t['type']['name'] for t in data['types']]
weight = data['weight']
    
print(f"\nName: {name.title()}")
print(f"Type: {', '.join(types)}")
print(f"Weight: {weight}")



Name: Pikachu
Type: electric
Weight: 60


## 3. The API Workflow: Chained Requests
Real-world data is rarely in one place. Often, you get an ID or a URL in response #1, and you need to fetch it in response #2. 

**Scenario:** You are building a chatbot that explains Pokemon abilities.
1.  The Pokemon endpoint tells you the *name* of the ability (e.g., "Static") and gives you a *link*.
2.  It does NOT tell you what "Static" actually does.
3.  Your code needs to follow that link to get the description.


Step 1: Get the Ability URL for the first ability from the Pikachu data

* Create a new variable called first_ability to contains the data from the first ability
* Create a new variable called ability_url to contains the URL for the first ability
* Create a new variable called ability_name to contains the name of the first ability

In [6]:
# EX

first_ability = data['abilities'][0]
ability_name = first_ability['ability']['name']
ability_url = first_ability['ability']['url']

Step 2: Now do a .request() on the new URL from the first ability, 
remember to use the .json() method to get the data from the response. 

Assign the data to a variable called ability_data

In [7]:
# EX
ability_response = requests.get(ability_url)

if ability_response.status_code == 200:
    ability_data = ability_response.json()

Step 3: Extract the English description

The API has multiple descriptions under different languages, we'll need to extract the English description.

To do this we'll need to loop through the descriptions and check if the language is English.

Print out the name of the pokemon, the ability name and the description.

In [8]:
# EX
for entry in ability_data['effect_entries']:
     if entry['language']['name'] == 'en':
        description = entry['short_effect']
        print(f"\nFACT: When {name.title()} uses {ability_name}, the effect is: {description}")
        break


FACT: When Pikachu uses static, the effect is: Has a 30% chance of paralyzing attacking Pokémon on contact.


## 4. Exercise: The "Context Builder"

**Your Task:**
Write a function `get_pokemon_context(pokemon_name)` that:
1.  Fetches the Pokemon's data.
2.  Fetches the data for its **first ability** (chained request).
3.  Returns a formatted string suitable for outputting

**Target Output:**
`"Snorlax is a Normal type pokemon. Its main ability is Immunity, which prevents the Pokémon from being poisoned."`


Data analytics scripts often run once and stop. AI systems run 24/7. 
What happens if the user types `get_pokemon_context('agumon')`? (Agumon is a Digimon!).

**Take your function to the next level**
1.  Add `try/except` blocks.
2.  Check `if response.status_code == 404`.
3.  Return a clean error message: *"Error: Pokemon not found."* instead of crashing.

In [ ]:
# YOUR SOLUTION HERE